# BENZI — LoRA fine-tune on Colab (~2–3 hours)

**Runtime:** Menu → *Runtime* → *Change runtime type* → **T4 GPU**

This notebook:
1. Clones your repo (or upload `fyp-ml-demos` if private)
2. Prepares empathy dataset
3. LoRA trains **Qwen2.5-3B** (4-bit GPU, ~60 steps)
4. Merges weights
5. Zips download for Ollama on your Mac

After download, on Mac see `benzi-server/docs/COLAB_TRAIN.md`.

In [ ]:
# Check GPU
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → T4 GPU')

In [ ]:
# Clone repo (change branch/url if needed)
import os
REPO = 'https://github.com/sameedsaeed123/final-year-benzi.git'
if not os.path.isdir('final-year-benzi'):
    !git clone --depth 1 https://github.com/sameedsaeed123/final-year-benzi.git
%cd final-year-benzi/fyp-ml-demos

**Private repo?** Skip clone. Use left folder 📁 → Upload `fyp-ml-demos` zip, then:
```python
%cd /content/fyp-ml-demos
```

In [ ]:
!pip install -q -r requirements.txt -r requirements-finetune.txt bitsandbytes accelerate

In [ ]:
# Step 1 — dataset (~5 min)
!python finetune/prepare_dataset.py --max-total 1200

In [ ]:
# Step 2 — train 3B LoRA on GPU (~45–90 min)
!python finetune/train_qlora.py --model Qwen/Qwen2.5-3B-Instruct --max-steps 60 --max-length 384 --batch-size 1 --grad-accum 4

In [ ]:
# Lighter / faster option (~20 min) — uncomment instead of cell above:
# !python finetune/train_qlora.py --benzi-lite

In [ ]:
# Step 3 — merge (~10 min)
!python finetune/merge_lora.py --model Qwen/Qwen2.5-3B-Instruct

In [ ]:
# Step 4 — zip for download
import shutil
from google.colab import files

shutil.make_archive('benzi-empathetic-trained', 'zip', 'finetune/merged/benzi-empathetic-hf')
shutil.make_archive('benzi-lora-adapter', 'zip', 'finetune/adapters/benzi-lora')
print('Downloading merged model + adapter zip…')
files.download('benzi-empathetic-trained.zip')
files.download('benzi-lora-adapter.zip')

## On your Mac (after download)

1. Unzip `benzi-empathetic-trained.zip` to e.g. `~/benzi-models/benzi-empathetic-hf`
2. Create Ollama model:
```bash
cd benzi-models/benzi-empathetic-hf
cat > Modelfile << 'EOF'
FROM .
PARAMETER temperature 0.65
PARAMETER num_ctx 4096
SYSTEM You are BENZI AI — supportive wellness between therapy sessions. Not a therapist. Defer clinical questions to their therapist.
EOF
ollama create benzi-empathetic-trained -f Modelfile
```
3. In `benzi-server/.env`: `OLLAMA_MODEL=benzi-empathetic-trained`
4. Restart API + keep `RAG_ENABLED=true` for context.